In [80]:
import pandas as pd
from textblob import TextBlob
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer
nltk.download('vader_lexicon')
import swifter

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


KeyError: 'query-planning'

In [ ]:
stock_news = pd.read_csv("../data/raw_analyst_ratings.csv", index_col=0)
stock_news.head()

,headline,url,publisher,date,stock
0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 10:30:54-04:00,A
1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 10:45:20-04:00,A
2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 04:30:07-04:00,A
3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 12:45:06-04:00,A
4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 11:38:59-04:00,A


In [26]:
# 1. Define the list of tickers and the structure to hold the DataFrames
tickers = ['NVDA', 'AAPL', 'AMZN', 'GOOG', 'META', 'MSFT']
file_extension = ".csv"
stock_data = {}

## ○ Date Alignment: Ensure that both datasets (news and stock prices) are aligned by dates. This might involve normalizing timestamps.
#### Prepare and normalize timestamps for news data and stock data
#### and compine both news adn stock data together

In [27]:
# 2. Load each CSV file into a DataFrame and store it in the dictionary
for ticker in tickers:
    # Construct the expected file name (e.g., 'NVDA.csv')
    filename = "../data/" +ticker + file_extension
    
    try:
        # Load the CSV file:
        # - index_col='Date': Assumes your date column is named 'Date'
        # - parse_dates=True: Converts the date column into datetime objects
        df = pd.read_csv(
            filename, 
            index_col='Date', 
            parse_dates=True
        )
        
        # Store the DataFrame in the dictionary
        stock_data[ticker] = df
        
        print(f" Loaded {filename}. Shape: {df.shape}")
        
    except FileNotFoundError:
        print(f" ERROR: File '{filename}' not found. Check your file path.")
    except Exception as e:
        print(f" ERROR processing {filename}: {e}")

 Loaded ../data/NVDA.csv. Shape: (3774, 5)
 Loaded ../data/AAPL.csv. Shape: (3774, 5)
 Loaded ../data/AMZN.csv. Shape: (3774, 5)
 Loaded ../data/GOOG.csv. Shape: (3774, 5)
 Loaded ../data/META.csv. Shape: (2923, 5)
 Loaded ../data/MSFT.csv. Shape: (3774, 5)


In [63]:
stock_data['AMZN'].head()

,Date,Close,High,Low,Open,Volume,date_only
0,2009-01-02,2.718,2.7265,2.5535,2.5675,145928000,2009-01-02
1,2009-01-05,2.703,2.7870,2.6515,2.7865,190196000,2009-01-05
2,2009-01-06,2.868,2.9110,2.6875,2.7275,221602000,2009-01-06
3,2009-01-07,2.810,2.8475,2.7675,2.8145,158854000,2009-01-07
4,2009-01-08,2.858,2.8660,2.7290,2.7495,131558000,2009-01-08


In [31]:
# Work with a copy of the stock_news DataFrame
news = stock_news.copy()

# Convert timestamp to datetime
news['date'] = pd.to_datetime(
    news['date'], 
    format='mixed', 
    utc=True, # Standardizes all times to UTC (essential for merging daily data)
    errors='coerce' # Converts unparseable dates to NaT (Not a Time)
)

# Normalize to date (removes time)
news['date_only'] = news['date'].dt.date

news.head()

,headline,url,publisher,date,stock,date_only
0,Stocks That Hit 52-Week Highs On Friday,https://www.benzinga.com/news/20/06/16190091/s...,Benzinga Insights,2020-06-05 14:30:54+00:00,A,2020-06-05
1,Stocks That Hit 52-Week Highs On Wednesday,https://www.benzinga.com/news/20/06/16170189/s...,Benzinga Insights,2020-06-03 14:45:20+00:00,A,2020-06-03
2,71 Biggest Movers From Friday,https://www.benzinga.com/news/20/05/16103463/7...,Lisa Levin,2020-05-26 08:30:07+00:00,A,2020-05-26
3,46 Stocks Moving In Friday's Mid-Day Session,https://www.benzinga.com/news/20/05/16095921/4...,Lisa Levin,2020-05-22 16:45:06+00:00,A,2020-05-22
4,B of A Securities Maintains Neutral on Agilent...,https://www.benzinga.com/news/20/05/16095304/b...,Vick Meyer,2020-05-22 15:38:59+00:00,A,2020-05-22


In [37]:
for ticker, df in stock_data.items():
    data = stock_data[ticker]
    # Reset index so Date becomes a column
    data = data.reset_index()
    # Normalize date to match news format
    data['date_only'] = data['Date'].dt.date
    stock_data[ticker] = data


In [75]:
news_and_stock_data = {}
for ticker in tickers:
    stock_df = stock_data.get(ticker)
    if stock_df is not None:
        merged_df = pd.merge(
            news, 
            stock_df, 
            how='inner', 
            on='date_only', 
            suffixes=('_news', f'_{ticker}')
        )
        news_and_stock_data[ticker] = merged_df
        print(f"Merged data for {ticker}. Shape: {merged_df.shape}")
    else:
        print(f"No stock data available for {ticker}.")


Merged data for NVDA. Shape: (1379682, 12)
Merged data for AAPL. Shape: (1379682, 12)
Merged data for AMZN. Shape: (1379682, 12)
Merged data for GOOG. Shape: (1379682, 12)
Merged data for META. Shape: (1111659, 12)
Merged data for MSFT. Shape: (1379682, 12)


In [74]:
news_and_stock_data['AMZN'].tail(20)

,Date,Close,High,Low,Open,Volume,date_only,headline,url,publisher,date,stock
1380679,2023-12-01,147.029999,147.250000,145.550003,146.000000,39951800,2023-12-01,NaN,NaN,NaN,NaT,NaN
1380680,2023-12-04,144.839996,145.350006,142.809998,145.250000,48294200,2023-12-04,NaN,NaN,NaN,NaT,NaN
1380681,2023-12-05,146.880005,148.570007,143.130005,143.550003,46822400,2023-12-05,NaN,NaN,NaN,NaT,NaN
1380682,2023-12-06,144.520004,147.850006,144.279999,147.580002,39679000,2023-12-06,NaN,NaN,NaN,NaT,NaN
1380683,2023-12-07,146.880005,147.919998,145.339996,146.149994,52352800,2023-12-07,NaN,NaN,NaN,NaT,NaN
1380684,2023-12-08,147.419998,147.839996,145.399994,145.479996,41906000,2023-12-08,NaN,NaN,NaN,NaT,NaN
1380685,2023-12-11,145.889999,146.190002,143.639999,145.660004,50907300,2023-12-11,NaN,NaN,NaN,NaT,NaN
1380686,2023-12-12,147.479996,147.500000,145.300003,145.520004,44944300,2023-12-12,NaN,NaN,NaN,NaT,NaN
1380687,2023-12-13,148.839996,149.460007,146.820007,148.119995,52766200,2023-12-13,NaN,NaN,NaN,NaT,NaN
1380688,2023-12-14,147.419998,150.539993,145.520004,149.929993,58400800,2023-12-14,NaN,NaN,NaN,NaT,NaN


## ○ Sentiment Analysis:
* We will assign a polarity score to each headline:
* polarity > 0 → positive
* polarity < 0 → negative
* polarity == 0 → neutral

In [ ]:
def get_sentiment(headline):
    analysis = TextBlob(str(headline))
    polarity = analysis.sentiment.polarity
    if polarity > 0:
        sentiment = 'positive'
    elif polarity < 0:
        sentiment = 'negative'
    else:
        sentiment = 'neutral'
    return pd.Series([polarity, sentiment])

#### Apply Sentiment Analysis per Ticker

In [59]:
news_and_stock_data['AAPL'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1379682 entries, 0 to 1379681
Data columns (total 12 columns):
 #   Column     Non-Null Count    Dtype              
---  ------     --------------    -----              
 0   headline   1379682 non-null  object             
 1   url        1379682 non-null  object             
 2   publisher  1379682 non-null  object             
 3   date       1379682 non-null  datetime64[ns, UTC]
 4   stock      1379682 non-null  object             
 5   date_only  1379682 non-null  object             
 6   Date       1379682 non-null  datetime64[ns]     
 7   Close      1379682 non-null  float64            
 8   High       1379682 non-null  float64            
 9   Low        1379682 non-null  float64            
 10  Open       1379682 non-null  float64            
 11  Volume     1379682 non-null  int64              
dtypes: datetime64[ns, UTC](1), datetime64[ns](1), float64(4), int64(1), object(5)
memory usage: 126.3+ MB


In [57]:
for ticker in tickers:
    df = news_and_stock_data[ticker]
    df[['sentiment_score', 'sentiment_label']] = df['headline'].swifter.apply(get_vader_sentiment)
    news_and_stock_data[ticker] = df

AttributeError: 'Series' object has no attribute 'swifter'